## Silver Stage 1: Clean, Cast, Deduplicate, Quarantine
Bronze data is raw strings. Here we cast proper types, fix messy fields (currency symbols, "unknown" prices, junk dates), split each source into clean + quarantine tables, and deduplicate on the business key.

In [0]:
%run "./00_setup"

%md
### Step 0: Setup
Creating catalog, schemas (raw/silver/gold), and a volume to store raw files.

### Helper Functions
`dedup_latest` keeps the most recently ingested row per business key. `clean_numeric` strips non-numeric characters (like "unknown" or currency symbols) and casts to double.

In [0]:
from pyspark.sql import functions as F, Window

def dedup_latest(df, key_cols, order_col="ingestion_ts"):
    w = Window.partitionBy(*key_cols).orderBy(F.col(order_col).desc())
    return (
        df.withColumn("_rn", F.row_number().over(w))
          .filter("_rn = 1")
          .drop("_rn")
    )

def clean_numeric(col):
    stripped = F.regexp_replace(F.col(col), r"[^0-9.\-]", "")
    return F.when(stripped == "", None).otherwise(stripped.cast("double"))

%md
### Path Configuration
Defining reusable paths for batch and incremental data, plus checkpoint/schema locations needed for streaming ingestion later.

### Orders: Union Batch + Incremental, Clean, Cast, Quarantine
Combining historical and incremental orders. `coupon_code` only exists from day 3 onward, so we align columns first. Bad rows (missing IDs, unparseable price/amount) go to a quarantine table instead of being deleted.

In [0]:
orders_batch = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_orders")
orders_incr = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_orders_incremental")

if "coupon_code" not in orders_batch.columns:
    orders_batch = orders_batch.withColumn("coupon_code", F.lit(None).cast("string"))
if "coupon_code" not in orders_incr.columns:
    orders_incr = orders_incr.withColumn("coupon_code", F.lit(None).cast("string"))
if "ingest_date" not in orders_batch.columns:
    orders_batch = orders_batch.withColumn("ingest_date", F.lit(None).cast("string"))

orders_raw = orders_batch.unionByName(orders_incr, allowMissingColumns=True)

orders_typed = (
    orders_raw
    .withColumn("order_ts", F.expr("try_to_timestamp(order_ts)"))
    .withColumn("quantity", F.expr("try_cast(quantity AS INT)"))
    .withColumn("unit_price", clean_numeric("unit_price"))
    .withColumn("discount_pct", F.expr("try_cast(discount_pct AS DOUBLE)"))
    .withColumn("gross_amount", clean_numeric("gross_amount"))
)

orders_quarantine = orders_typed.filter(
    F.col("order_id").isNull()
    | F.col("customer_id").isNull()
    | F.col("product_id").isNull()
    | F.col("order_ts").isNull()
    | F.col("unit_price").isNull()
    | F.col("gross_amount").isNull()
)

orders_clean = orders_typed.subtract(orders_quarantine)
orders_clean = dedup_latest(orders_clean, ["order_id"])

orders_clean.write.format("delta").mode("overwrite").option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean")
orders_quarantine.write.format("delta").mode("overwrite").option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_quarantine")

print("orders clean:", orders_clean.count(), " | quarantined:", orders_quarantine.count())

orders clean: 17273  | quarantined: 734


### Customers: Clean, Cast, Quarantine
Parse signup_date, fill missing categorical fields with "Unknown" instead of dropping them, quarantine rows with no customer_id, and dedup on customer_id.

In [0]:
customers_batch = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_customers")
customers_typed = (
    customers_batch
    .withColumn("signup_date", F.expr("try_to_date(signup_date, 'yyyy-MM-dd')"))
    .withColumn("city", F.when(F.col("city").isNull() | (F.col("city") == ""), "Unknown").otherwise(F.col("city")))
    .withColumn("segment", F.when(F.col("segment").isNull() | (F.col("segment") == ""), "Unknown").otherwise(F.col("segment")))
    .withColumn("gender", F.when(F.col("gender").isNull() | (F.col("gender") == ""), "Unknown").otherwise(F.col("gender")))
)

customers_quarantine = customers_typed.filter(
    F.col("customer_id").isNull() | (F.col("customer_id") == "")
)
customers_clean = customers_typed.subtract(customers_quarantine)
customers_clean = dedup_latest(customers_clean, ["customer_id"])

customers_clean.write.format("delta").mode("overwrite").option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver1_customers_clean")
customers_quarantine.write.format("delta").mode("overwrite").option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver1_customers_quarantine")

print("customers clean:", customers_clean.count(), " | quarantined:", customers_quarantine.count())

customers clean: 2500  | quarantined: 0


### Products: Clean, Cast, Quarantine
Clean unit_price (handles "unknown" values), parse created_date, fill missing category/brand with "Unknown", quarantine rows with no product_id or unparseable price.

In [0]:
products_batch = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_products")
products_typed = (
    products_batch
    .withColumn("unit_price", clean_numeric("unit_price"))
    .withColumn("created_date", F.expr("try_to_date(created_date, 'yyyy-MM-dd')"))
    .withColumn("category", F.when(F.col("category").isNull() | (F.col("category") == ""), "Unknown").otherwise(F.col("category")))
    .withColumn("brand", F.when(F.col("brand").isNull() | (F.col("brand") == ""), "Unknown").otherwise(F.col("brand")))
)

products_quarantine = products_typed.filter(
    F.col("product_id").isNull() | F.col("unit_price").isNull()
)
products_clean = products_typed.subtract(products_quarantine)
products_clean = dedup_latest(products_clean, ["product_id"])

products_clean.write.format("delta").mode("overwrite").option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver1_products_clean")
products_quarantine.write.format("delta").mode("overwrite").option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver1_products_quarantine")

print("products clean:", products_clean.count(), " | quarantined:", products_quarantine.count())

products clean: 780  | quarantined: 20


### Stores: Deduplicate
Small reference dimension — just dedup on store_id, no complex cleaning needed.

In [0]:
stores_batch = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_stores")
stores_clean = dedup_latest(stores_batch, ["store_id"])

stores_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.dim_store")

print("stores clean:", stores_clean.count())

stores clean: 75
